# 07 — Run summarized-experience memory

This repeats the 12-task Vertex run with one concise, model-generated summary stored after every answer. Retrieval is relevance top-1 and occurs only on turns 2–3. Token and latency totals include summary generation.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / 'src' / 'state_management').is_dir() else WORKING_DIR.parent
EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
RUNS_DIR = EXPERIMENTS_DIR / 'runs'
MANIFEST_PATH = RUNS_DIR / 'tasks.jsonl'
RESULTS_PATH = RUNS_DIR / 'summary_results.jsonl'
MEMORY_PATH = RUNS_DIR / 'summaries.sqlite3'
RUNNER_PATH = EXPERIMENTS_DIR / 'run_vertex_memory_condition.py'
print('Summaries: 24 paid requests: 12 task answers plus 12 summaries.')

In [ ]:
environment = os.environ.copy()
environment.setdefault('VERTEX_AI_PROJECT_ID', 'dsi-tifin-harness')
environment.setdefault('VERTEX_AI_LOCATION', 'global')
environment.setdefault('VERTEX_MODEL', 'gemini-3.5-flash-lite')
run = subprocess.run([sys.executable, str(RUNNER_PATH), 'summary', str(MANIFEST_PATH), str(RESULTS_PATH), str(MEMORY_PATH)], cwd=PROJECT_ROOT, env=environment, text=True, capture_output=True, check=False)
print(run.stdout)
if run.returncode:
    print(run.stderr)
    raise RuntimeError('Summary run stopped; completed records were preserved.')

In [ ]:
results = [json.loads(line) for line in RESULTS_PATH.read_text(encoding='utf-8').splitlines() if line]
assert len(results) == 12
retrievals = [x for x in results if x['turn'] in (2, 3)]
print('Completed:', len(results), '| Same-episode relevance proxy:', sum(x['retrieval_relevant'] is True for x in retrievals), '/', len(retrievals))